In [3]:
#Install dependencies
!pip install great-expectations==0.18.21 numpy==1.26.4 pandas==2.2.2 pytest -q

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [19 lines of output]
      + C:\Users\monir\anaconda3\envs\MLOps\python.exe C:\Users\monir\AppData\Local\Temp\pip-install-tb8bjtjv\numpy_25c36bd0039c440e9ed250331f4f1b12\vendored-meson\meson\meson.py setup C:\Users\monir\AppData\Local\Temp\pip-install-tb8bjtjv\numpy_25c36bd0039c440e9ed250331f4f1b12 C:\Users\monir\AppData\Local\Temp\pip-install-tb8bjtjv\numpy_25c36bd0039c440e9ed250331f4f1b12\.mesonpy-rbpsbxr1 -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\monir\AppData\Local\Temp\pip-install-tb8bjtjv\numpy_25c36bd0039c440e9ed250331f4f1b12\.mesonpy-rbpsbxr1\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\monir\AppData\Local\Temp\pip-install-tb8bjtjv\numpy_25c36bd0039c440e9ed250331f4f1b12
      Build dir: C:\Users\monir\AppData\Local\Temp\pip-install-tb8bjtjv\numpy_25c36bd

In [4]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)
n = 700


#

In [ ]:
# ── Path configuration ─────────────────────────────────────────────────────────
# os.getcwd() returns the folder where this notebook lives.
# We build all paths relative to it so the notebook works on any machine.
BASE_DIR   = os.getcwd()
DATA_PATH  = os.path.join(BASE_DIR, "data", "customer_data.csv")
GE_DIR     = os.path.join(BASE_DIR, "gx")          # GE stores its config here

print("Base directory :", BASE_DIR)
print("Dataset path   :", DATA_PATH)
print("GE directory   :", GE_DIR)

In [5]:
import great_expectations as gx

# Create an in-memory DataContext
print("DataContext loaded from:", context.root_directory)
print("Great Expectations DataContext initialized!")


ModuleNotFoundError: No module named 'great_expectations'

In [ ]:
# ── Register a Pandas CSV datasource ──────────────────────────────────────────
# A 'datasource' tells GE where the data lives and what engine reads it.
# PandasFilesystemDatasource is the simplest option: it reads local CSV files
# into a pandas DataFrame, which is exactly what we need.

DATASOURCE_NAME = "customer_csv_datasource"

# add_or_update_pandas_filesystem_datasource is idempotent: safe to re-run
datasource = context.sources.add_or_update_pandas_filesystem_datasource(
    name=DATASOURCE_NAME,
    base_directory=os.path.join(BASE_DIR, "data"),
)

# A 'data asset' is a specific file (or table) within the datasource
asset = datasource.add_csv_asset(
    name="customer_data",
    filepath_or_buffer="customer_data.csv",
)

# A 'batch request' specifies the exact slice of data to validate
batch_request = asset.build_batch_request()
print("Datasource and asset registered successfully.")

In [ ]:
# ── Create the expectation suite ───────────────────────────────────────────────
# An expectation suite is a named collection of validation rules.
# add_or_update_expectation_suite is idempotent.
SUITE_NAME = "customer_data_expectations"

suite = context.add_or_update_expectation_suite(expectation_suite_name=SUITE_NAME)
print(f"Expectation suite '{SUITE_NAME}' ready.")

In [ ]:
# ── Create a Validator bound to our CSV batch ──────────────────────────────────
# The Validator loads the data into memory and attaches the expectation suite.
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=SUITE_NAME,
)
print("Validator created. Columns:", list(validator.active_batch.data.dataframe.columns))

In [ ]:
# ── Expectation 1: customer_id – must not be null ──────────────────────────────
# WHY: customer_id is our primary key. A null key means we cannot identify the
#      record, which would corrupt joins and aggregations downstream.
result_1a = validator.expect_column_values_to_not_be_null(column="customer_id")
print("customer_id not-null:", result_1a["success"])

In [ ]:
# ── Expectation 2: customer_id – must be unique ────────────────────────────────
# WHY: Duplicate IDs indicate duplicated customer records. Training a model on
#      duplicated rows inflates the weight of those records and skews results.
result_1b = validator.expect_column_values_to_be_unique(column="customer_id")
print("customer_id unique:", result_1b["success"])

In [ ]:
# ── Expectation 3: age – must be between 0 and 120 ────────────────────────────
# WHY: Ages outside 0–120 are biologically impossible and are data entry errors.
#      They would appear as outliers and distort any age-based feature.
result_2 = validator.expect_column_values_to_be_between(
    column="age",
    min_value=0,
    max_value=120,
)
print("age in [0, 120]:", result_2["success"])

In [ ]:
# ── Expectation 4: email – must match a valid email regex ─────────────────────
# WHY: Invalid emails (missing '@', no domain, etc.) cannot be used for
#      communication or as a reliable join key with other tables.
# HOW: We use a standard RFC-5321-inspired regex. The 'mostly' parameter is NOT
#      used here because even one invalid email is a data problem worth flagging.
EMAIL_REGEX = r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,6}$"

result_3 = validator.expect_column_values_to_match_regex(
    column="email",
    regex=EMAIL_REGEX,
)
print("email format valid:", result_3["success"])

In [ ]:
# ── Expectation 5: salary – must be present in at least 95 % of rows ──────────
# WHY: A small fraction of missing salaries is acceptable (the assignment
#      explicitly allows this), but if more than 5 % are missing the column
#      becomes unreliable for modelling and imputation would introduce too much
#      noise.
# HOW: 'mostly=0.95' means GE passes the check if ≥95 % of values are non-null.
result_4 = validator.expect_column_values_to_not_be_null(
    column="salary",
    mostly=0.95,
)
print("salary present ≥95%:", result_4["success"])

In [ ]:
# ── Expectation 6: country – must be one of four allowed values ────────────────
# WHY: Countries outside the allowed set ('India', 'Spain', 'Brazil', …) are
#      either data entry errors or records from unsupported markets that should
#      be excluded before modelling.
result_5 = validator.expect_column_values_to_be_in_set(
    column="country",
    value_set=["USA", "Canada", "UK", "Australia"],
)
print("country in allowed set:", result_5["success"])

In [ ]:
# ── Expectation 7: signup_date – must be parseable as a datetime ───────────────
# WHY: If signup_date is stored as a string that cannot be parsed, any time-
#      series feature (tenure, seasonality) will fail silently or produce NaT.
# HOW: expect_column_values_to_be_dateutil_parseable uses Python's dateutil
#      library to check every value. It catches garbage strings like '2023-02-30'
#      (Feb 30 does not exist) and genuinely unparseable entries.
result_6 = validator.expect_column_values_to_be_dateutil_parseable(
    column="signup_date"
)
print("signup_date parseable:", result_6["success"])

In [ ]:
# ── Expectation 8: row count – must be between 500 and 1000 ───────────────────
# WHY: A drastically different row count compared to what we expect indicates
#      that the data pipeline dropped rows (too few) or duplicated the entire
#      file (too many). Both scenarios should halt processing.
result_7 = validator.expect_table_row_count_to_be_between(
    min_value=500,
    max_value=1000,
)
print("row count in [500, 1000]:", result_7["success"])

In [ ]:
# ── Save all expectations to disk ─────────────────────────────────────────────
# save_expectation_suite() writes the suite to
# gx/expectations/customer_data_expectations.json
# so it can be reused in future runs without redefining each expectation.
validator.save_expectation_suite(discard_failed_expectations=False)
print("Expectation suite saved.")